<a href="https://colab.research.google.com/github/rashmikanaujiya1701/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/blob/main/04_FastAPI_Churn_Service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04_FastAPI_Churn_Service

**Name:** Rashmi Kanaujiya

**Student ID:** (iitp_aiml_25061009)  
**Course:** Trimester 3 Project  
**Dataset:** d2c churn data package

### train_model.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
base_path = "/content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("/content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/data/rfm_modeling_snapshot.csv")

target = "churn_next_60d"

X = df.drop(columns=[target])
y = df[target]

for col in X.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

model.fit(X_train, y_train)

joblib.dump(model, "model.pkl")

print("model.pkl saved")

model.pkl saved


## app/schemas.py

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class CustomerFeatures(BaseModel):

    recency_days: float = Field(..., ge=0)
    frequency_180d: float = Field(..., ge=0)
    monetary_180d: float = Field(..., ge=0)

    ticket_count_90d: float = Field(..., ge=0)

    sessions_30d: float = Field(..., ge=0)

    campaign_clicks_30d: float = Field(..., ge=0)


class BatchRequest(BaseModel):
    customers: List[CustomerFeatures]

In [ ]:
import os

# Create the 'app' directory if it doesn't exist
if not os.path.exists('app'):
    os.makedirs('app')
    print("Created directory: app/")
else:
    print("Directory 'app/' already exists.")

Directory 'app/' already exists.


In [ ]:
schemas_content = """
from pydantic import BaseModel, Field
from typing import List

class CustomerFeatures(BaseModel):

    recency_days: float = Field(..., ge=0)
    frequency_180d: float = Field(..., ge=0)
    monetary_180d: float = Field(..., ge=0)

    ticket_count_90d: float = Field(..., ge=0)

    sessions_30d: float = Field(..., ge=0)

    campaign_clicks_30d: float = Field(..., ge=0)


class BatchRequest(BaseModel):
    customers: List[CustomerFeatures]
"""

with open("app/schemas.py", "w", encoding="utf-8") as f:
    f.write(schemas_content.strip())

print("app/schemas.py created successfully!")

app/schemas.py created successfully!


In [ ]:
main_content = """
from fastapi import FastAPI

from app.schemas import (
    CustomerFeatures,
    BatchRequest
)

import joblib
import numpy as np

app = FastAPI(
    title="Customer Churn API"
)

model = joblib.load("model.pkl")


@app.get("/health")
def health():

    return {
        "status": "ok"
    }


def risk_text(prob):

    if prob >= 0.70:
        return (
            "Low recent activity and "
            "high support activity "
            "indicate elevated churn risk."
        )

    elif prob >= 0.40:
        return (
            "Customer shows moderate "
            "risk indicators."
        )

    return (
        "Customer appears relatively "
        "stable."
    )


@app.post("/predict")
def predict(customer: CustomerFeatures):

    features = np.array([
        [
            customer.recency_days,
            customer.frequency_180d,
            customer.monetary_180d,
            customer.ticket_count_90d,
            customer.sessions_30d,
            customer.campaign_clicks_30d
        ]
    ])

    prob = float(
        model.predict_proba(features)[0][1]
    )

    pred = int(prob >= 0.40)

    risk_level = (
        "high"
        if prob >= 0.70
        else "medium"
        if prob >= 0.40
        else "low"
    )

    return {

        "churn_probability":
        round(prob,4),

        "predicted_class":
        pred,

        "risk_level":
        risk_level,

        "risk_explanation":
        risk_text(prob)

    }


@app.post("/batch_predict")
def batch_predict(
    request: BatchRequest
):

    results = []

    for customer in request.customers:

        features = np.array([
            [
                customer.recency_days,
                customer.frequency_180d,
                customer.monetary_180d,
                customer.ticket_count_90d,
                customer.sessions_30d,
                customer.campaign_clicks_30d
            ]
        ])

        prob = float(
            model.predict_proba(features)[0][1]
        )

        pred = int(prob >= 0.40)

        results.append({

            "churn_probability":
            round(prob,4),

            "predicted_class":
            pred,

            "risk_level":
            (
                "high"
                if prob >= 0.70
                else "medium"
                if prob >= 0.40
                else "low"
            )

        })

    return {
        "predictions": results
    }
"""

with open("app/main.py", "w", encoding="utf-8") as f:
    f.write(main_content.strip())

print("app/main.py created successfully!")

app/main.py created successfully!


app/main.py

In [ ]:
import sys
sys.path.insert(0, '')

from fastapi import FastAPI

from app.schemas import (
    CustomerFeatures,
    BatchRequest
)

import joblib
import numpy as np

app = FastAPI(
    title="Customer Churn API"
)

model = joblib.load("model.pkl")


@app.get("/health")
def health():

    return {
        "status": "ok"
    }


def risk_text(prob):

    if prob >= 0.70:
        return (
            "Low recent activity and "
            "high support activity "
            "indicate elevated churn risk."
        )

    elif prob >= 0.40:
        return (
            "Customer shows moderate "
            "risk indicators."
        )

    return (
        "Customer appears relatively "
        "stable."
    )


@app.post("/predict")
def predict(customer: CustomerFeatures):

    features = np.array([
        [
            customer.recency_days,
            customer.frequency_180d,
            customer.monetary_180d,
            customer.ticket_count_90d,
            customer.sessions_30d,
            customer.campaign_clicks_30d
        ]
    ])

    prob = float(
        model.predict_proba(features)[0][1]
    )

    pred = int(prob >= 0.40)

    risk_level = (
        "high"
        if prob >= 0.70
        else "medium"
        if prob >= 0.40
        else "low"
    )

    return {

        "churn_probability":
        round(prob,4),

        "predicted_class":
        pred,

        "risk_level":
        risk_level,

        "risk_explanation":
        risk_text(prob)

    }


@app.post("/batch_predict")
def batch_predict(
    request: BatchRequest
):

    results = []

    for customer in request.customers:

        features = np.array([
            [
                customer.recency_days,
                customer.frequency_180d,
                customer.monetary_180d,
                customer.ticket_count_90d,
                customer.sessions_30d,
                customer.campaign_clicks_30d
            ]
        ])

        prob = float(
            model.predict_proba(features)[0][1]
        )

        pred = int(prob >= 0.40)

        results.append({

            "churn_probability":
            round(prob,4),

            "predicted_class":
            pred,

            "risk_level":
            (
                "high"
                if prob >= 0.70
                else "medium"
                if prob >= 0.40
                else "low"
            )

        })

    return {
        "predictions": results
    }

tests/test_api.py

In [ ]:
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)


def test_health():

    response = client.get(
        "/health"
    )

    assert response.status_code == 200

    assert (
        response.json()["status"]
        == "ok"
    )


def test_predict():

    payload = {

        "recency_days": 60,
        "frequency_180d": 2,
        "monetary_180d": 1000,

        "ticket_count_90d": 4,

        "sessions_30d": 1,

        "campaign_clicks_30d": 0
    }

    response = client.post(
        "/predict",
        json=payload
    )

    assert response.status_code == 200

    assert (
        "churn_probability"
        in response.json()
    )


def test_batch_predict():

    payload = {

        "customers":[

            {
                "recency_days":60,
                "frequency_180d":2,
                "monetary_180d":1000,
                "ticket_count_90d":4,
                "sessions_30d":1,
                "campaign_clicks_30d":0
            },

            {
                "recency_days":5,
                "frequency_180d":20,
                "monetary_180d":15000,
                "ticket_count_90d":0,
                "sessions_30d":15,
                "campaign_clicks_30d":8
            }

        ]
    }

    response = client.post(
        "/batch_predict",
        json=payload
    )

    assert response.status_code == 200

In [ ]:
{
  "recency_days": 45,
  "frequency_180d": 3,
  "monetary_180d": 2500,
  "ticket_count_90d": 2,
  "sessions_30d": 4,
  "campaign_clicks_30d": 1
}

{'recency_days': 45,
 'frequency_180d': 3,
 'monetary_180d': 2500,
 'ticket_count_90d': 2,
 'sessions_30d': 4,
 'campaign_clicks_30d': 1}

In [ ]:
{
  "churn_probability": 0.72,
  "predicted_class": 1,
  "risk_level": "high",
  "risk_explanation": "Low recent activity and high support activity indicate elevated churn risk."
}

{'churn_probability': 0.72,
 'predicted_class': 1,
 'risk_level': 'high',
 'risk_explanation': 'Low recent activity and high support activity indicate elevated churn risk.'}

In [ ]:
schemas_content = """
from pydantic import BaseModel
from typing import List

class CustomerFeatures(BaseModel):

    recency: float
    frequency: float
    monetary: float

    support_ticket_count: float

    sessions_30d: float
    campaign_clicks_30d: float


class BatchRequest(BaseModel):
    customers: List[CustomerFeatures]
"""

with open("app/schemas.py", "w", encoding="utf-8") as f:
    f.write(schemas_content)

print("app/schemas.py created successfully!")

app/schemas.py created successfully!


In [ ]:
main_content = """
from fastapi import FastAPI
from app.schemas import (
    CustomerFeatures,
    BatchRequest
)

import joblib
import numpy as np

app = FastAPI(
    title="Customer Churn API"
)

model = joblib.load("model.pkl")


@app.get("/health")
def health():

    return {
        "status": "ok"
    }


def risk_text(prob):

    if prob >= 0.70:
        return (
            "Low recent activity and "
            "high support activity "
            "indicate elevated churn risk."
        )

    elif prob >= 0.40:
        return (
            "Customer shows moderate "
            "risk indicators."
        )

    return (
        "Customer appears relatively "
        "stable."
    )


@app.post("/predict")
def predict(customer: CustomerFeatures):

    features = np.array([
        [
            customer.recency,
            customer.frequency,
            customer.monetary,
            customer.support_ticket_count,
            customer.sessions_30d,
            customer.campaign_clicks_30d
        ]
    ])

    prob = float(
        model.predict_proba(features)[0][1]
    )

    pred = int(prob >= 0.40)

    risk_level = (
        "high"
        if prob >= 0.70
        else "medium"
        if prob >= 0.40
        else "low"
    )

    return {

        "churn_probability":
        round(prob,4),

        "predicted_class":
        pred,

        "risk_level":
        risk_level,

        "risk_explanation":
        risk_text(prob)

    }


@app.post("/batch_predict")
def batch_predict(
    request: BatchRequest
):

    results = []

    for customer in request.customers:

        features = np.array([
            [
                customer.recency,
                customer.frequency,
                customer.monetary,
                customer.support_ticket_count,
                customer.sessions_30d,
                customer.campaign_clicks_30d
            ]
        ])

        prob = float(
            model.predict_proba(features)[0][1]
        )

        pred = int(prob >= 0.40)

        results.append({

            "churn_probability":
            round(prob,4),

            "predicted_class":
            pred,

            "risk_level":
            (
                "high"
                if prob >= 0.70
                else "medium"
                if prob >= 0.40
                else "low"
            )

        })

    return {
        "predictions": results
    }
"""

with open("app/main.py", "w", encoding="utf-8") as f:
    f.write(main_content)

print("app/main.py created successfully!")

app/main.py created successfully!


monitoring_plan.md

In [ ]:
monitoring_plan = """
# Monitoring Plan & Responsible Use Guide

## Overview

This document defines the monitoring strategy for the Customer Churn Prediction API after deployment.

The objective is to ensure:

- Model reliability
- Prediction quality
- API stability
- Business effectiveness

---

# 1. Data Drift Monitoring

## Objective

Detect changes in incoming customer data compared with training data.

## Features to Monitor

### RFM Features

- Recency
- Frequency
- Monetary Value

### Customer Support Features

- Support Ticket Count
- Complaint Frequency

### Engagement Features

- Sessions in Last 30 Days
- Campaign Clicks
- Product Views

## Monitoring Frequency

- Weekly monitoring
- Monthly drift analysis

## Alert Conditions

Generate alerts when:

- Feature mean changes by more than 20%
- Significant distribution shifts occur
- Missing values increase unexpectedly

## Business Impact

Data drift may reduce prediction quality and increase incorrect churn predictions.

---

# 2. Prediction Distribution Monitoring

## Objective

Track prediction patterns over time.

## Metrics

Monitor:

- Average churn probability
- Percentage of High-Risk customers
- Percentage of Medium-Risk customers
- Percentage of Low-Risk customers

## Alert Conditions

Investigate if:

- High-risk predictions increase suddenly
- Probability distributions shift significantly
- Predictions differ greatly from training behavior

---

# 3. Business Outcome Monitoring

## Objective

Measure business value created by the model.

## Metrics

### Retention Metrics

- Retention Rate
- Churn Rate
- Win-back Rate

### Campaign Metrics

- Campaign Response Rate
- Offer Acceptance Rate
- Conversion Rate

### Financial Metrics

- Revenue Retained
- Customer Lifetime Value
- Cost Per Retained Customer

## Monitoring Frequency

Monthly

---

# 4. API Monitoring

## Objective

Ensure FastAPI service reliability.

## Metrics

### Availability

- API Uptime
- Service Availability %

### Performance

- Average Response Time
- P95 Latency
- Request Throughput

### Error Tracking

- HTTP 4xx Errors
- HTTP 5xx Errors
- Validation Errors
- Failed Prediction Requests

## Alert Conditions

- Error Rate > 5%
- API Downtime
- High Response Latency

---

# 5. Model Performance Monitoring

## Objective

Track predictive performance after deployment.

## Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- ROC-AUC

## Priority Metric

Recall

Reason:
Missing churners creates higher business loss than targeting extra customers.

## Alert Conditions

- Recall drops below target
- ROC-AUC declines significantly
- False negatives increase

---

# 6. Retraining Strategy

## Retraining Triggers

Retrain when:

- Significant Data Drift occurs
- Recall decreases substantially
- Business KPIs decline
- Customer behavior changes

## Schedule

- Monthly Performance Review
- Quarterly Full Retraining
- Emergency Retraining if major drift is detected

---

# 7. Responsible Use Guidelines

## Appropriate Use

Use predictions for:

- Customer Retention Campaigns
- Marketing Prioritization
- Customer Success Outreach
- Churn Risk Assessment

Predictions should support decisions, not replace human judgment.

## Human Review Required

Manual review is recommended for:

- High-Value Customers
- Ambiguous Cases
- Expensive Retention Offers

---

# 8. Inappropriate Use

This API must NOT be used for:

- Credit Approval Decisions
- Employment Decisions
- Legal Decisions
- Insurance Decisions
- High-Impact Decisions affecting individual rights

The model is intended only for churn prediction and retention planning.

---

# 9. Monitoring Ownership

| Area | Responsible Team |
|--------|------------------|
| API Health | Engineering Team |
| Data Quality | Data Team |
| Model Performance | Data Science Team |
| Business Outcomes | Marketing Team |
| Retraining | Data Science Team |

---

# Conclusion

Continuous monitoring of data quality, prediction performance, API health, and business outcomes is essential to maintain an accurate and reliable churn prediction service.
"""

with open("monitoring_plan.md", "w", encoding="utf-8") as f:
    f.write(monitoring_plan)

print("monitoring_plan.md created successfully!")

monitoring_plan.md created successfully!


requirements.txt

In [ ]:
requirements_content = """
fastapi==0.115.0
uvicorn==0.30.6
pydantic==2.9.2
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.5.1
joblib==1.4.2
pytest==8.3.3
httpx==0.27.2
"""

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content.strip())

print("requirements.txt created successfully!")

requirements.txt created successfully!


README.md

In [ ]:
readme_content = """
# Customer Churn Prediction API

## Project Overview

This project provides a FastAPI-based churn prediction service that can be integrated with CRM systems.

The API loads a trained machine learning model and returns:

- Churn Probability
- Predicted Class
- Risk Level
- Risk Explanation

The service is designed to support customer retention teams by identifying customers who are likely to churn.

---

## Repository Structure

Part4_FastAPI_Churn_Service/

├── README.md
├── requirements.txt
├── model.pkl
├── train_model.py
├── monitoring_plan.md

├── app/
│   ├── main.py
│   └── schemas.py

└── tests/
    └── test_api.py

---

## Dataset

Training data source:

- rfm_modeling_snapshot.csv

Additional customer behavior datasets:

- orders.csv
- support_tickets.csv
- web_events_snapshot.csv
- intervention_history.csv

---

## Install Dependencies

```bash
pip install -r requirements.txt
```

---

## Train Model

```bash
python train_model.py
```

Output:

```text
model.pkl
```

---

## Run API

```bash
uvicorn app.main:app --reload
```

Open Swagger UI:

```text
http://127.0.0.1:8000/docs
```

---

## Endpoints

### GET /health

Response:

```json
{
  "status": "ok"
}
```

### POST /predict

Returns:

* churn_probability
* predicted_class
* risk_level
* risk_explanation

### POST /batch_predict

Returns predictions for multiple customers.

---

## Running Tests

```bash
pytest
```

---

## Model Information

Algorithm: Random Forest Classifier

Threshold: 0.40

---

## Monitoring

See:

* monitoring_plan.md

---

## Responsible Use

Use for:

* Customer Retention
* Marketing Prioritization
* Customer Success

Do NOT use for:

* Credit Decisions
* Employment Decisions
* Legal Decisions
* Insurance Decisions
  """

with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("README.md created successfully!")

README.md created successfully!


In [ ]:
import os
import shutil

target_directory = os.path.join(base_path, 'D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/')

# Create the target directory if it doesn't exist
if not os.path.exists(target_directory):
    os.makedirs(target_directory)
    print(f"Created directory: {target_directory}")
else:
    print(f"Directory '{target_directory}' already exists.")

Directory '/content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/' already exists.


In [ ]:
# Write README.md
readme_file_path = os.path.join(target_directory, "README.md")
with open(readme_file_path, "w", encoding="utf-8") as f:
    f.write(readme_content)
print(f"README.md created at {readme_file_path}")

README.md created at /content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/README.md


In [ ]:
# Write requirements.txt
requirements_file_path = os.path.join(target_directory, "requirements.txt")
with open(requirements_file_path, "w", encoding="utf-8") as f:
    f.write(requirements_content.strip())
print(f"requirements.txt created at {requirements_file_path}")

requirements.txt created at /content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/requirements.txt


In [ ]:
# Write monitoring_plan.md
monitoring_file_path = os.path.join(target_directory, "monitoring_plan.md")
with open(monitoring_file_path, "w", encoding="utf-8") as f:
    f.write(monitoring_plan)
print(f"monitoring_plan.md created at {monitoring_file_path}")

monitoring_plan.md created at /content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/monitoring_plan.md


In [ ]:
# Copy model.pkl
source_model_path = "model.pkl"
dest_model_path = os.path.join(target_directory, "model.pkl")
shutil.copy(source_model_path, dest_model_path)
print(f"model.pkl copied from {source_model_path} to {dest_model_path}")

model.pkl copied from model.pkl to /content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/model.pkl


In [ ]:
import os
import subprocess

# Define the outputs directory path
outputs_directory = os.path.join(target_directory, 'outputs')

# Create the outputs directory if it doesn't exist
if not os.path.exists(outputs_directory):
    os.makedirs(outputs_directory)
    print(f"Created directory: {outputs_directory}")
else:
    print(f"Directory '{outputs_directory}' already exists.")

Directory '/content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/outputs' already exists.


In [ ]:
# Run pytest and save the output to a file in the outputs directory
test_report_path = os.path.join(outputs_directory, "test_report.txt")

# Ensure the current working directory is where pytest can find 'app' and 'tests'
# Assuming 'app' and 'tests' are in /content/
original_cwd = os.getcwd()
os.chdir('/content/')

try:
    with open(test_report_path, "w") as f:
        # Using subprocess to run pytest and redirect its stdout to the file
        process = subprocess.run(
            ["pytest"],
            capture_output=True,
            text=True,
            check=True # Raise an exception for non-zero exit codes
        )
        f.write(process.stdout)
        if process.stderr:
            f.write("\n--- STDERR ---\n")
            f.write(process.stderr)
    print(f"pytest results saved to {test_report_path}")
except subprocess.CalledProcessError as e:
    print(f"pytest command failed with error code {e.returncode}")
    print(f"STDOUT:\n{e.stdout}")
    print(f"STDERR:\n{e.stderr}")
    with open(test_report_path, "w") as f:
        f.write(f"pytest command failed with error code {e.returncode}\n")
        f.write(f"STDOUT:\n{e.stdout}\n")
        f.write(f"STDERR:\n{e.stderr}\n")
except FileNotFoundError:
    print("pytest command not found. Please ensure pytest is installed.")
finally:
    os.chdir(original_cwd) # Restore original working directory

pytest command failed with error code 5
STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: anyio-4.13.0, typeguard-4.5.2, langsmith-0.8.9
collected 0 items

============================ no tests ran in 1.39s =============================

STDERR:



In [ ]:
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load the model (already loaded but for completeness)
# model = joblib.load("model.pkl")

# Make predictions on the test set
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

metrics = {
    "accuracy": round(accuracy, 4),
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "f1_score": round(f1, 4),
    "roc_auc": round(roc_auc, 4)
}

# Define the path for metrics.json
metrics_file_path = os.path.join(outputs_directory, "metrics.json")

# Save metrics to metrics.json
with open(metrics_file_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=4)

print(f"Metrics saved to {metrics_file_path}")
print(json.dumps(metrics, indent=4))

Metrics saved to /content/drive/MyDrive/T3_Project_Rashmi_Kanaujiya/D2C_Customer_Churn_Intelligence/Notebooks/FastAPI_Churn_Scoring_Service_Reproducible_ML_Workflow/outputs/metrics.json
{
    "accuracy": 0.7917,
    "precision": 0.799,
    "recall": 0.7422,
    "f1_score": 0.7696,
    "roc_auc": 0.8494
}


# FastAPI Churn Scoring Service & Reproducible ML Workflow — Project Summary

This project develops a production-ready FastAPI-based churn prediction service that allows internal CRM and retention systems to obtain real-time customer churn-risk predictions. The API loads a trained machine learning model and exposes endpoints for both single-customer and batch churn scoring.

A Random Forest Classifier is used as the final prediction model and is stored as model.pkl.

API Endpoints
GET /health

Performs a health check to verify that the API is running correctly.

POST /predict

Accepts a single customer record and returns:

Churn probability
Predicted class
Risk level
Risk explanation
POST /batch_predict

Accepts multiple customer records and returns churn predictions for each customer.

Input Validation

The API uses Pydantic models to validate incoming requests and ensure data consistency before scoring.

